# Cleaning_CYRV_customers_dataset

## Conclusion

The customer dataset was found to be structurally clean and consistent.

No missing values, duplicate rows, duplicate `customer_id` values, or clear formatting errors were identified. Repeated `customer_unique_id` values were retained because they represent customers associated with multiple orders.

Customer location information was validated against the geolocation reference dataset. Geographic mismatches were not automatically corrected because the reference data does not provide sufficient evidence that the customer values are incorrect.

The final dataset preserves all 99,441 original customer records and is ready for further analysis and integration with the orders and geolocation datasets.

## Data Cleaning Summary

The `CYRV_customers_dataset.csv` dataset was inspected and validated for missing values, duplicate records, identifier consistency, ZIP code prefixes, city and state formatting, and geographic consistency.

### Dataset Overview
- Total rows: 99,441
- Total columns: 5
- Missing values: 0
- Duplicate rows: 0

### Customer Identifiers
- `customer_id` contains 99,441 unique values and no duplicates.
- `customer_unique_id` contains 96,096 unique values.
- Repeated `customer_unique_id` values were retained because the same customer can be associated with multiple orders and therefore multiple `customer_id` values.
- 2,997 customers are associated with more than one `customer_id`.

### Customer Location
- `customer_zip_code_prefix` contains 14,994 unique ZIP code prefixes.
- Four-digit ZIP prefixes were retained as integers because leading zeros are not preserved in the source numeric representation.
- `customer_state` contains 27 valid two-character uppercase state codes.
- No invalid state formatting, leading/trailing spaces, or empty values were found.
- `customer_city` contains 4,119 unique values.
- No leading/trailing spaces, uppercase values, empty strings, or multiple consecutive spaces were found.
- City names containing apostrophes and hyphens were reviewed and retained as valid geographic names.

### Geolocation Validation
The customer location data was cross-validated against `CYRV_geolocation_dataset.csv` using ZIP code prefix, state, and normalized city names.

- 99,139 customer records matched the geolocation reference by ZIP code, state, and city.
- 302 customer records did not fully match the geolocation reference.
- Of these, 278 records had ZIP/state combinations not available in the geolocation dataset.
- 24 records had valid ZIP/state combinations but different city names.
- These city differences were reviewed and retained because they may represent districts, localities, or alternative geographic naming rather than incorrect customer data.
- No geographic values were automatically replaced based solely on the geolocation reference.

### Repeated Customer Locations
Among customers associated with multiple `customer_id` values:
- 250 customers had more than one ZIP code.
- 122 customers had more than one city.
- 39 customers had more than one state.

These records were retained because a customer's location may differ between orders.

### Cleaning Actions
No source records required modification or removal. Temporary normalized city values used for validation were removed before the final dataset was saved.

In [15]:
import pandas as pd

customers = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_customers_dataset.csv")

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [16]:
print("Shape:", customers.shape)

print("\nData types:")
print(customers.dtypes)

Shape: (99441, 5)

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object


In [17]:
missing_values = customers.isna().sum()

missing_percent = (
    customers.isna().mean() * 100
).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percent": missing_percent
})

missing_summary.sort_values(
    "missing_count",
    ascending=False
)

,missing_count,missing_percent
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


In [18]:
print("Duplicate rows:", customers.duplicated().sum())

Duplicate rows: 0


In [19]:
print("Total rows:", len(customers))

print("\nCustomer ID:")
print("Unique customer IDs:", customers["customer_id"].nunique())
print(
    "Duplicated customer IDs:",
    customers["customer_id"].duplicated().sum()
)

print("\nCustomer Unique ID:")
print(
    "Unique customer unique IDs:",
    customers["customer_unique_id"].nunique()
)
print(
    "Repeated customer unique ID rows:",
    customers["customer_unique_id"].duplicated().sum()
)

Total rows: 99441

Customer ID:
Unique customer IDs: 99441
Duplicated customer IDs: 0

Customer Unique ID:
Unique customer unique IDs: 96096
Repeated customer unique ID rows: 3345


In [20]:
customers["customer_unique_id"].value_counts().value_counts().sort_index()

count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

In [21]:
print("Min ZIP prefix:", customers["customer_zip_code_prefix"].min())
print("Max ZIP prefix:", customers["customer_zip_code_prefix"].max())
print("Unique ZIP prefixes:", customers["customer_zip_code_prefix"].nunique())

print("\nZIP prefix digit lengths:")
print(
    customers["customer_zip_code_prefix"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
)

Min ZIP prefix: 1003
Max ZIP prefix: 99990
Unique ZIP prefixes: 14994

ZIP prefix digit lengths:
customer_zip_code_prefix
4    23995
5    75446
Name: count, dtype: int64


In [23]:
print("Unique states:", customers["customer_state"].nunique())

print("\nState values:")
print(
    customers["customer_state"]
    .value_counts()
    .sort_index()
)

print(
    "\nInvalid state length:",
    customers["customer_state"].str.len().ne(2).sum()
)

print(
    "Leading/trailing spaces:",
    customers["customer_state"]
    .ne(customers["customer_state"].str.strip())
    .sum()
)

print(
    "Contains lowercase:",
    customers["customer_state"]
    .str.contains(r"[a-z]", regex=True)
    .sum()
)

print(
    "Empty/whitespace-only:",
    customers["customer_state"]
    .str.strip()
    .eq("")
    .sum()
)

Unique states: 27

State values:
customer_state
AC       81
AL      413
AM      148
AP       68
BA     3380
CE     1336
DF     2140
ES     2033
GO     2020
MA      747
MG    11635
MS      715
MT      907
PA      975
PB      536
PE     1652
PI      495
PR     5045
RJ    12852
RN      485
RO      253
RR       46
RS     5466
SC     3637
SE      350
SP    41746
TO      280
Name: count, dtype: int64

Invalid state length: 0
Leading/trailing spaces: 0
Contains lowercase: 0
Empty/whitespace-only: 0


In [24]:
print("Unique cities:", customers["customer_city"].nunique())

print(
    "Leading/trailing spaces:",
    customers["customer_city"]
    .ne(customers["customer_city"].str.strip())
    .sum()
)

print(
    "Contains uppercase:",
    customers["customer_city"]
    .str.contains(r"[A-Z]", regex=True)
    .sum()
)

print(
    "Empty/whitespace-only:",
    customers["customer_city"]
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Contains multiple spaces:",
    customers["customer_city"]
    .str.contains(r"\s{2,}", regex=True)
    .sum()
)

Unique cities: 4119
Leading/trailing spaces: 0
Contains uppercase: 0
Empty/whitespace-only: 0
Contains multiple spaces: 0


In [25]:
special_char_mask = customers["customer_city"].str.contains(
    r"[^a-z0-9\s]",
    regex=True
)

print(
    "Rows with special characters:",
    special_char_mask.sum()
)

print(
    "Unique cities with special characters:",
    customers.loc[
        special_char_mask,
        "customer_city"
    ].nunique()
)

print("\nCity values:")
print(
    sorted(
        customers.loc[
            special_char_mask,
            "customer_city"
        ].unique()
    )
)

Rows with special characters: 450
Unique cities with special characters: 52

City values:
["alta floresta d'oeste", "alvorada d'oeste", "aparecida d'oeste", "arraial d'ajuda", "bandeirantes d'oeste", 'biritiba-mirim', 'ceara-mirim', 'cipo-guacu', "conquista d'oeste", "diamante d'oeste", "dias d'avila", 'embu-guacu', 'entre-ijuis', "estrela d'oeste", "figueiropolis d'oeste", 'governador dix-sept rosado', 'guajara-mirim', 'guarda-mor', "herval d'oeste", 'igarape-acu', 'igarape-miri', "itapejara d'oeste", "itaporanga d'ajuda", 'ji-parana', "lagoa d'anta", "machadinho d'oeste", "mae d'agua", "mirassol d'oeste", 'mogi-guacu', 'mogi-mirim', 'nao-me-toque', "nova brasilandia d'oeste", "olho d'agua das cunhas", "olho d'agua das flores", "olhos d'agua", "palmeira d'oeste", 'pariquera-acu', "pau d'arco", "perola d'oeste", "pingo-d'agua", "rancho alegre d'oeste", "santa barbara d'oeste", "santa clara d'oeste", "santa rita d'oeste", "sao joao d'alianca", "sao jorge d'oeste", 'sapucai-mirim', "tanq

In [26]:
geolocation = pd.read_csv(
    "../../data/cyrv/CYRV_geolocation_dataset.csv"
)

print("Shape:", geolocation.shape)

print("\nData types:")
print(geolocation.dtypes)

print("\nMissing values:")
print(geolocation.isna().sum())

Shape: (1000163, 5)

Data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Missing values:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64


In [27]:
geo_zip_state = geolocation[
    ["geolocation_zip_code_prefix", "geolocation_state"]
].drop_duplicates()

customer_zip_state = customers[
    ["customer_zip_code_prefix", "customer_state"]
].drop_duplicates()

zip_state_check = customer_zip_state.merge(
    geo_zip_state,
    left_on=["customer_zip_code_prefix", "customer_state"],
    right_on=["geolocation_zip_code_prefix", "geolocation_state"],
    how="left",
    indicator=True
)

print("Unique customer ZIP/state combinations:", len(customer_zip_state))

print(
    "ZIP/state found in geolocation:",
    (zip_state_check["_merge"] == "both").sum()
)

print(
    "ZIP/state not found in geolocation:",
    (zip_state_check["_merge"] == "left_only").sum()
)

Unique customer ZIP/state combinations: 14994
ZIP/state found in geolocation: 14837
ZIP/state not found in geolocation: 157


In [28]:
import unicodedata

def normalize_city(value):
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        char for char in value
        if not unicodedata.combining(char)
    )
    return value.strip().lower()


customers["city_normalized"] = (
    customers["customer_city"]
    .apply(normalize_city)
)

geolocation["city_normalized"] = (
    geolocation["geolocation_city"]
    .apply(normalize_city)
)

print(
    "Unique normalized customer cities:",
    customers["city_normalized"].nunique()
)

print(
    "Unique normalized geolocation cities:",
    geolocation["city_normalized"].nunique()
)

Unique normalized customer cities: 4119
Unique normalized geolocation cities: 5967


In [29]:
geo_reference = geolocation[
    [
        "geolocation_zip_code_prefix",
        "geolocation_state",
        "city_normalized"
    ]
].drop_duplicates()

customer_geo_check = customers.merge(
    geo_reference,
    left_on=[
        "customer_zip_code_prefix",
        "customer_state",
        "city_normalized"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_state",
        "city_normalized"
    ],
    how="left",
    indicator=True
)

print("Total customer rows:", len(customers))

print(
    "Matched ZIP/state/city:",
    (customer_geo_check["_merge"] == "both").sum()
)

print(
    "Not matched ZIP/state/city:",
    (customer_geo_check["_merge"] == "left_only").sum()
)

Total customer rows: 99441
Matched ZIP/state/city: 99139
Not matched ZIP/state/city: 302


In [30]:
geo_zip_state_reference = set(
    zip(
        geolocation["geolocation_zip_code_prefix"],
        geolocation["geolocation_state"]
    )
)

unmatched_customers = customer_geo_check[
    customer_geo_check["_merge"] == "left_only"
].copy()

unmatched_customers["zip_state_found"] = unmatched_customers.apply(
    lambda row: (
        row["customer_zip_code_prefix"],
        row["customer_state"]
    ) in geo_zip_state_reference,
    axis=1
)

print("Total unmatched customer rows:", len(unmatched_customers))

print(
    "ZIP/state not found in geolocation:",
    (~unmatched_customers["zip_state_found"]).sum()
)

print(
    "ZIP/state found but city differs:",
    unmatched_customers["zip_state_found"].sum()
)

Total unmatched customer rows: 302
ZIP/state not found in geolocation: 278
ZIP/state found but city differs: 24


In [32]:
city_mismatches = unmatched_customers[
    unmatched_customers["zip_state_found"]
][
    [
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].drop_duplicates()

geo_city_options = geolocation[
    [
        "geolocation_zip_code_prefix",
        "geolocation_state",
        "geolocation_city"
    ]
].drop_duplicates()

city_mismatch_details = city_mismatches.merge(
    geo_city_options,
    left_on=[
        "customer_zip_code_prefix",
        "customer_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="left"
)

city_mismatch_details[
    [
        "customer_zip_code_prefix",
        "customer_state",
        "customer_city",
        "geolocation_city"
    ]
].sort_values(
    [
        "customer_state",
        "customer_zip_code_prefix",
        "customer_city"
    ]
)

,customer_zip_code_prefix,customer_state,customer_city,geolocation_city
11,45534,BA,taboquinhas,itacare
8,48793,BA,caldas do jorro,tucano
0,48967,BA,nucleo residencial pilar,jaguarari
2,71090,DF,brasilia,guara
1,73251,DF,brasilia,sobradinho
9,29386,ES,piacu,muniz freire
3,38749,MG,sao benedito,sao joao da serra negra
4,38749,MG,sao benedito,silvano
5,38749,MG,sao benedito,patrocinio
6,68514,PA,morada nova,maraba


In [33]:
mismatch_counts = (
    unmatched_customers[
        unmatched_customers["zip_state_found"]
    ]
    .groupby(
        [
            "customer_zip_code_prefix",
            "customer_state",
            "customer_city"
        ]
    )
    .size()
    .reset_index(name="customer_count")
    .sort_values(
        "customer_count",
        ascending=False
    )
)

mismatch_counts

,customer_zip_code_prefix,customer_state,customer_city,customer_count
7,48967,BA,nucleo residencial pilar,8
10,73251,DF,brasilia,3
1,15903,SP,guariroba,2
9,71090,DF,brasilia,2
11,85138,PR,colonia jordaozinho,2
0,13508,SP,ajapi,1
2,28143,RJ,mussurepe,1
3,29386,ES,piacu,1
4,38749,MG,sao benedito,1
5,45534,BA,taboquinhas,1


In [34]:
customer_location_counts = (
    customers
    .groupby("customer_unique_id")
    .agg(
        customer_ids=("customer_id", "nunique"),
        zip_codes=("customer_zip_code_prefix", "nunique"),
        cities=("customer_city", "nunique"),
        states=("customer_state", "nunique")
    )
)

repeated_customers = customer_location_counts[
    customer_location_counts["customer_ids"] > 1
]

print("Customers with multiple customer IDs:", len(repeated_customers))

print(
    "With multiple ZIP codes:",
    (repeated_customers["zip_codes"] > 1).sum()
)

print(
    "With multiple cities:",
    (repeated_customers["cities"] > 1).sum()
)

print(
    "With multiple states:",
    (repeated_customers["states"] > 1).sum()
)

Customers with multiple customer IDs: 2997
With multiple ZIP codes: 250
With multiple cities: 122
With multiple states: 39


In [35]:
customers = customers.drop(
    columns=["city_normalized"]
)

print("Shape:", customers.shape)

print("\nMissing values:")
print(customers.isna().sum())

print("\nDuplicate rows:", customers.duplicated().sum())

print(
    "Unique customer IDs:",
    customers["customer_id"].nunique()
)

print("\nData types:")
print(customers.dtypes)

Shape: (99441, 5)

Missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Duplicate rows: 0
Unique customer IDs: 99441

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object


In [37]:
customers.to_csv(
    "../CYRV_customers_cleaned.csv",
    index=False
)

print("Cleaned customers dataset saved successfully.")

Cleaned customers dataset saved successfully.


In [38]:
customers_check = pd.read_csv(
    "../CYRV_customers_cleaned.csv"
)

print("Shape:", customers_check.shape)
print("Missing values:", customers_check.isna().sum().sum())
print("Duplicate rows:", customers_check.duplicated().sum())
print("Unique customer IDs:", customers_check["customer_id"].nunique())

print("\nData types:")
print(customers_check.dtypes)

Shape: (99441, 5)
Missing values: 0
Duplicate rows: 0
Unique customer IDs: 99441

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object
